In [5]:
using Serialization
using Oscar
using ProgressMeter
using SparseArrays
using Serialization
using Base.Threads
using LinearAlgebra
using Nemo

const F2 = GF(2)


function boundary_incidence_facets_to_ridges(facets::Vector{UInt16})
    # collect ridges (each facet contributes its (d-1)-subfaces by deleting one vertex)
    ridge_dict = Dict{UInt16, Int}()  # ridge -> row index
    ridges = Vector{UInt16}()
    for f in facets
        for i=0:((8*sizeof(f))-1)
            if (f>>i)&1==1
                r = f ⊻ (UInt16(1)<<i)
                if !haskey(ridge_dict, r)
                    push!(ridges, r)
                    ridge_dict[r] = length(ridges)
                end
            end
        end
    end

    m = length(ridges)
    n = length(facets)

    # build sparse 0/1 matrix (m x n): ridges x facets
    I = Int[]   # row indices
    J = Int[]   # col indices

    for (j, f) in pairs(facets)
        for i=0:(8*sizeof(f)-1)
            if (f>>i)&1==1
                r = f ⊻ (UInt16(1)<<i)
                row_idx = ridge_dict[r]  
                push!(I, row_idx)
                push!(J, j)
            end
        end
    end

    A = sparse(I, J, true, m, n)  # SparseMatrixCSC{Bool} 

    return ridges, A
end

function mod2_rank_nemo(A::SparseMatrixCSC)
    m, n = size(A)
    if m == 0 || n == 0
        return 0
    end

    M = zero_matrix(F2, m, n)

    # Fill matrix directly from sparse structure
    for col in 1:n
        for idx in A.colptr[col]:(A.colptr[col+1]-1)
            row = A.rowval[idx]
            M[row, col] = F2(1)
        end
    end

    return rank(M)
end


function is_mod2_sphere(top_facets::Vector{UInt16})

    isempty(top_facets) && return true

    d = count_ones(top_facets[1]) - 1

    current_faces = top_facets

    # ---- Step 1: Top homology β_d ----
    # β_d = (#d-faces - rank ∂_d)
    ridges, B = boundary_incidence_facets_to_ridges(current_faces)
    rank_prev = mod2_rank_nemo(B)

    n_d = length(current_faces)
    β_d = n_d - rank_prev
    β_d == 1 || return false

    current_faces = ridges

    # ---- Step 2: Middle dimensions ----
    # For i = d-1 down to 1:
    for dim = d-1:-1:1

        ridges, B = boundary_incidence_facets_to_ridges(current_faces)
        rank_curr = mod2_rank_nemo(B)

        n_i = length(current_faces)

        # β_i = (#i-faces - rank ∂_i) - rank ∂_{i+1}
        β_i = (n_i - rank_curr) - rank_prev
        β_i == 0 || return false

        rank_prev = rank_curr
        current_faces = ridges
    end

    # ---- Step 3: β_0 ----
    # β_0 = (#vertices) - rank ∂_1
    n_0 = length(current_faces)
    β_0 = n_0 - rank_prev
    β_0 == 1 || return false

    return true
end
function is_seed(MNF::Vector{Set{Int}}, m::Int)
    MNF_sets = Set(MNF)

    # iterate over all pairs of vertices
    for v in 1:m-1
        for w in v+1:m

            pair = Set((v, w))
            is_pair = true

            for F in MNF_sets
                # exactly one present → break condition
                if (v in F) ⊻ (w in F)
                    is_pair = false
                    break
                end
            end

            # if always together and pair not itself MNF → not seed
            if is_pair && !(pair in MNF_sets)
                return false
            end
        end
    end

    return true
end



is_seed (generic function with 1 method)

In [ ]:
data_PLS_5_9 = [parse.(Int, split(replace(line, r"[\[\]\s]" => ""), ',')) for line in readlines("CSPLS_5_9.txt")]


mat_DB = open("rank_4_simple_bin_mat_DB_bin_test.jls", "r") do io
    deserialize(io)
end

pseudo_manifolds_DB = open("Pic_4_DB_6-15_test.jls", "r") do io
    deserialize(io)
end



list_isom_new = Vector{Vector{Vector{Int}}}()

for (l,bases) in enumerate(mat_DB[9])
    # display(bases)
    V = reduce(|,bases)
    compl_bases = [base⊻V for base in bases]
    @showprogress for facets_bit in pseudo_manifolds_DB[9][l]
        facets_L_bin = compl_bases[findall(facets_bit)]
        facets_L = [[i for i=1:(8*sizeof(facet_bin)) if (facet_bin>>(i-1))&1==1] for facet_bin in facets_L_bin]
        L = simplicial_complex(facets_L)

        is_isom=false
        for facets_M in list_isom_new
            if is_isomorphic(L,simplicial_complex(facets_M))
                is_isom=true
                break
            end
        end
        if !is_isom
            push!(list_isom_new,facets_L)
        end
    end
end

for facets_bin in data_PLS_5_9
    facets_K = [[i for i=1:(8*sizeof(facet_bin)) if (facet_bin>>(i-1))&1==1] for facet_bin in facets_bin]
    K = simplicial_complex(facets_K)
    is_isom=false
    for facets_L in list_isom_new
        L = simplicial_complex(facets_L)
        if is_isomorphic(K,L)
            is_isom=true
            break
        end
    end
    if !is_isom
        display(facets_K)
    end
end


Progress: 100%|█████████████████████████████████████████| Time: 0:00:10
Progress: 100%|█████████████████████████████████████████| Time: 0:03:43
Progress: 100%|█████████████████████████████████████████| Time: 0:08:01
Progress: 100%|█████████████████████████████████████████| Time: 0:01:45
Progress: 100%|█████████████████████████████████████████| Time: 0:03:27


30-element Vector{Vector{Int64}}:
 [1, 2, 3, 4, 5]
 [1, 2, 3, 5, 6]
 [1, 3, 4, 5, 6]
 [1, 2, 4, 5, 7]
 [2, 3, 4, 5, 7]
 [1, 3, 4, 6, 7]
 [2, 3, 5, 6, 7]
 [1, 4, 5, 6, 7]
 [3, 4, 5, 6, 7]
 [1, 2, 3, 6, 8]
 [1, 2, 5, 6, 8]
 [1, 3, 4, 7, 8]
 [1, 3, 6, 7, 8]
 ⋮
 [1, 5, 6, 7, 9]
 [2, 5, 6, 7, 9]
 [1, 2, 3, 8, 9]
 [1, 3, 4, 8, 9]
 [1, 2, 5, 8, 9]
 [1, 5, 6, 8, 9]
 [2, 5, 6, 8, 9]
 [2, 3, 7, 8, 9]
 [1, 4, 7, 8, 9]
 [3, 4, 7, 8, 9]
 [1, 6, 7, 8, 9]
 [2, 6, 7, 8, 9]